In [40]:
import numpy as np
import pandas as pd
from pathlib import Path

import optuna
from lightgbm import LGBMClassifier
from xgboost import XGBClassifier
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import balanced_accuracy_score
from sklearn.preprocessing import FunctionTransformer
from sklearn.pipeline import Pipeline

# other
from meteocalc import dew_point

In [41]:
DATA_DIR = Path("../../data/kaggle-irrigation")
SEED = 42

# Feature Engineering Pipeline

In [42]:
def add_features(d: pd.DataFrame):
    d = d.copy()

    d["heat_stress"]   = (d["Temperature_C"] * d["Sunlight_Hours"]).astype(int)
    d["drying_force"]  = (d["Wind_Speed_kmh"] * d["Temperature_C"] / (d["Humidity"] + 1)).astype(int)
    d["soil_quality"]  = (d["Organic_Carbon"] / (d["Electrical_Conductivity"] + 0.1)).astype(int)
    d["dew_point"] = d.apply(lambda row: dew_point(row["Temperature_C"], row["Humidity"]).c, axis=1)

    return d

In [43]:
feature_pipe = FunctionTransformer(add_features)

# Get Training Data

In [44]:
df = pd.read_csv(DATA_DIR / "train.csv", index_col="id")
df.head()

,Soil_Type,Soil_pH,Soil_Moisture,Organic_Carbon,Electrical_Conductivity,Temperature_C,Humidity,Rainfall_mm,Sunlight_Hours,Wind_Speed_kmh,Crop_Type,Crop_Growth_Stage,Season,Irrigation_Type,Water_Source,Field_Area_hectare,Mulching_Used,Previous_Irrigation_mm,Region,Irrigation_Need
id,,,,,,,,,,,,,,,,,,,,
0,Loamy,4.92,32.58,1.01,3.05,15.01,50.61,725.99,5.90,16.79,Sugarcane,Sowing,Zaid,Drip,Rainwater,0.82,No,112.16,East,Low
1,Clay,7.08,56.61,0.44,2.00,22.92,67.86,985.66,6.98,3.39,Wheat,Vegetative,Kharif,Rainfed,River,5.27,Yes,47.16,South,Low
2,Clay,5.69,27.71,0.81,2.83,26.97,92.22,2201.70,6.05,3.85,Rice,Vegetative,Kharif,Sprinkler,Reservoir,8.24,Yes,110.38,North,Low
3,Sandy,5.65,13.32,1.33,0.87,13.32,61.57,1357.33,9.12,2.31,Wheat,Flowering,Kharif,Canal,River,8.32,Yes,53.85,South,Medium
4,Clay,7.96,59.14,0.38,0.96,20.22,91.11,1538.20,6.95,13.94,Wheat,Sowing,Rabi,Canal,River,7.37,No,93.19,South,Low


In [45]:
# convert dtypes for categorical handling
cat_cols = df.select_dtypes(include=["str", "object"]).columns
df[cat_cols] = df[cat_cols].astype("category")

In [46]:
X = df.drop(columns=["Irrigation_Need"])
y = df["Irrigation_Need"]

le = LabelEncoder()
y_encoded = le.fit_transform(y)

X_train, X_test, y_train, y_test = train_test_split(X, y_encoded, test_size=0.2, shuffle=True, stratify=y, random_state=SEED)
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)

# Benchmarking

## Model 1 - XGBoost

In [47]:
xgb = XGBClassifier(
    random_state=SEED,
    enable_categorical=True,
    device="cuda:0",
    tree_method="hist"
)

base_xgb = Pipeline(
    [
        ("features", feature_pipe),
        ("xgb", xgb)
    ]
)

base_xgb_cvs = cross_val_score(base_xgb, X_train, y_train, cv=skf, n_jobs=-1, scoring="balanced_accuracy")
print(f"cv scores: {base_xgb_cvs.mean():.4f} +/- {base_xgb_cvs.std():.4f}")

base_xgb.fit(X_train, y_train)
base_xgb_preds = base_xgb.predict(X_test)

base_xgb_metric_score = balanced_accuracy_score(y_test, base_xgb_preds)
print(f"Balanced Accuracy: {base_xgb_metric_score:.4f}")

cv scores: 0.9616 +/- 0.0016
Balanced Accuracy: 0.9652


## Model 2 - LightGBM

In [48]:
base_lgbm = Pipeline(
    [
        ("features", feature_pipe),
        ("lgbm", LGBMClassifier(random_state=SEED, enable_categorical=True, n_jobs=-1))
    ]
)

base_lgbm_cvs = cross_val_score(base_lgbm, X_train, y_train, cv=skf, n_jobs=-1, scoring="balanced_accuracy")
print(f"cv scores: {base_lgbm_cvs.mean():.4f} +/- {base_lgbm_cvs.std():.4f}")

base_lgbm.fit(X_train, y_train)
base_lgbm_preds = base_lgbm.predict(X_test)

base_lgbm_metric_score = balanced_accuracy_score(y_test, base_lgbm_preds)
print(f"Balanced Accuracy: {base_lgbm_metric_score:.4f}")

cv scores: 0.9606 +/- 0.0013
[LightGBM] [Warning] Unknown parameter: enable_categorical
[LightGBM] [Warning] Unknown parameter: enable_categorical
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.004870 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 3252
[LightGBM] [Info] Number of data points in the train set: 504000, number of used features: 23
[LightGBM] [Info] Start training from score -3.400781
[LightGBM] [Info] Start training from score -0.532440
[LightGBM] [Info] Start training from score -0.968948
[LightGBM] [Warning] Unknown parameter: enable_categorical
Balanced Accuracy: 0.9650


## Evaluation

In [49]:
results = pd.DataFrame({
    "cv_mean": [base_xgb_cvs.mean(), base_lgbm_cvs.mean()],
    "cv_std": [base_xgb_cvs.std(), base_lgbm_cvs.std()],
    "balanced_accuracy": [base_xgb_metric_score, base_lgbm_metric_score],
}, index=["Base XGB", "Base LGBM"])

results.round(4)

,cv_mean,cv_std,balanced_accuracy
Base XGB,0.9616,0.0016,0.9652
Base LGBM,0.9606,0.0013,0.9650


# Tuning XGB

**Score to Beat**: 0.9616

## Tuning 1

`min_child_weight, max_depth, min_split_loss`

In [50]:
def objective(trial):
    min_child_weight  = trial.suggest_float("min_child_weight", 1, 10)
    max_depth = trial.suggest_int('max_depth', 10, 25)
    min_split_loss = trial.suggest_float("min_split_loss", 0, 5)

    cls_optuna = XGBClassifier(
                    random_state=42,
                    enable_categorical=True,
                    min_child_weight=min_child_weight,
                    max_depth=max_depth,
                    min_split_loss=min_split_loss,
                    device="cuda:0",
                    tree_method="hist"
                )

    pipe = Pipeline(
        [
            ("features", feature_pipe),
            ("model", cls_optuna)
        ]
    )

    cv_scores = cross_val_score(pipe, X_train, y_train, cv=skf, scoring='balanced_accuracy', n_jobs=-1)
    return cv_scores.mean()

study1 = optuna.create_study(direction='maximize')
study1.optimize(objective, n_trials=10)

# results
study_result = pd.DataFrame({
    "cv_mean": [study1.best_value],
    "cv_std": [np.nan],
    "balanced_accuracy": [np.nan]

}, index=["XGB Tuning 1"])

results = pd.concat([results, study_result])

[I 2026-04-22 22:22:32,573] A new study created in memory with name: no-name-21e032a5-7bbe-4408-9b82-f8d1b6159510
[I 2026-04-22 22:22:48,729] Trial 0 finished with value: 0.9609150485948434 and parameters: {'min_child_weight': 6.017186780764584, 'max_depth': 16, 'min_split_loss': 4.444428539569651}. Best is trial 0 with value: 0.9609150485948434.
[I 2026-04-22 22:23:01,411] Trial 1 finished with value: 0.9611676445414126 and parameters: {'min_child_weight': 7.29770976354905, 'max_depth': 11, 'min_split_loss': 3.9602455091448503}. Best is trial 1 with value: 0.9611676445414126.
[I 2026-04-22 22:23:14,404] Trial 2 finished with value: 0.9612337953496386 and parameters: {'min_child_weight': 7.253641518018302, 'max_depth': 12, 'min_split_loss': 3.643313268018445}. Best is trial 2 with value: 0.9612337953496386.
[I 2026-04-22 22:23:48,855] Trial 3 finished with value: 0.960526988546861 and parameters: {'min_child_weight': 7.734936078578772, 'max_depth': 20, 'min_split_loss': 0.3487799609097

## Tuning 2

`bagging_fraction, colsample_bytree`

In [51]:
def objective(trial):
    bagging_fraction = trial.suggest_float("bagging_fraction", 0.6, 1.0)
    colsample_bytree = trial.suggest_float("colsample_bytree", 0.3, 0.7)

    cls_optuna = XGBClassifier(
                    random_state=42,
                    enable_categorical=True,
                    bagging_fraction=bagging_fraction,
                    colsample_bytree=colsample_bytree,
                    device="cuda:0",
                    tree_method="hist",
                )

    pipe = Pipeline(
        [
            ("features", feature_pipe),
            ("model", cls_optuna)
        ]
    )

    cv_scores = cross_val_score(pipe, X_train, y_train, cv=skf, scoring='balanced_accuracy', n_jobs=-1)
    return cv_scores.mean()

study2 = optuna.create_study(direction='maximize')
study2.optimize(objective, n_trials=10)

# results
study_result = pd.DataFrame({
    "cv_mean": [study2.best_value],
    "cv_std": [np.nan],
    "balanced_accuracy": [np.nan]

}, index=["XGB Tuning 2"])

results = pd.concat([results, study_result])

[I 2026-04-22 22:26:00,592] A new study created in memory with name: no-name-98740e2a-5d4a-4713-9893-0b996202bcf7
[I 2026-04-22 22:26:17,266] Trial 0 finished with value: 0.9604119177237728 and parameters: {'bagging_fraction': 0.8255374438812276, 'colsample_bytree': 0.5415746969888461}. Best is trial 0 with value: 0.9604119177237728.
[I 2026-04-22 22:26:33,825] Trial 1 finished with value: 0.9600625757204044 and parameters: {'bagging_fraction': 0.6009884011458384, 'colsample_bytree': 0.4239122797244794}. Best is trial 0 with value: 0.9604119177237728.
[I 2026-04-22 22:26:50,411] Trial 2 finished with value: 0.9610829960984365 and parameters: {'bagging_fraction': 0.9192534573212561, 'colsample_bytree': 0.696853987734335}. Best is trial 2 with value: 0.9610829960984365.
[I 2026-04-22 22:27:07,093] Trial 3 finished with value: 0.9602131377410614 and parameters: {'bagging_fraction': 0.7519244153578051, 'colsample_bytree': 0.4763842916446561}. Best is trial 2 with value: 0.9610829960984365.

## Tuning 3

`learning_rate, n_estimators`

In [52]:
def objective(trial):
    learning_rate  = trial.suggest_float("learning_rate", 0.01, 0.4)
    n_estimators = trial.suggest_int("n_estimators", 100, 2000)

    cls_optuna = XGBClassifier(
                    random_state=42,
                    enable_categorical=True,
                    learning_rate=learning_rate,
                    n_estimators=n_estimators,
                    device="cuda:0",
                    tree_method="hist",
                )

    pipe = Pipeline(
        [
            ("features", feature_pipe),
            ("model", cls_optuna)
        ]
    )

    cv_scores = cross_val_score(pipe, X_train, y_train, cv=skf, scoring='balanced_accuracy', n_jobs=-1)
    return cv_scores.mean()

study3 = optuna.create_study(direction='maximize')
study3.optimize(objective, n_trials=10)

# results
study_result = pd.DataFrame({
    "cv_mean": [study3.best_value],
    "cv_std": [np.nan],
    "balanced_accuracy": [np.nan]

}, index=["XGB Tuning 3"])

results = pd.concat([results, study_result])

[I 2026-04-22 22:28:46,484] A new study created in memory with name: no-name-8152fce9-1f15-4168-9c38-544fe9bd6ad6
[I 2026-04-22 22:32:09,592] Trial 0 finished with value: 0.9614237177691708 and parameters: {'learning_rate': 0.03430097130225599, 'n_estimators': 1808}. Best is trial 0 with value: 0.9614237177691708.
[I 2026-04-22 22:35:32,672] Trial 1 finished with value: 0.9612399220375059 and parameters: {'learning_rate': 0.15447066700715525, 'n_estimators': 1806}. Best is trial 0 with value: 0.9614237177691708.
[I 2026-04-22 22:38:48,796] Trial 2 finished with value: 0.9611097408088993 and parameters: {'learning_rate': 0.2706037926223347, 'n_estimators': 1929}. Best is trial 0 with value: 0.9614237177691708.
[I 2026-04-22 22:39:59,106] Trial 3 finished with value: 0.9614389181825802 and parameters: {'learning_rate': 0.09340300071898047, 'n_estimators': 589}. Best is trial 3 with value: 0.9614389181825802.
[I 2026-04-22 22:41:33,590] Trial 4 finished with value: 0.9615938897401575 and 

## Bonus Tuning

In [56]:
def objective(trial):
    max_delta_step = trial.suggest_float("max_delta_step", 0, 10)

    cls_optuna = XGBClassifier(
                    random_state=42,
                    enable_categorical=True,
                    device="cuda:0",
                    tree_method="hist",
                    max_delta_step=max_delta_step,
                    **study1.best_params,
                    **study2.best_params,
                    **study3.best_params,
                )

    pipe = Pipeline(
        [
            ("features", feature_pipe),
            ("model", cls_optuna)
        ]
    )

    cv_scores = cross_val_score(pipe, X_train, y_train, cv=skf, scoring='balanced_accuracy', n_jobs=-1)
    return cv_scores.mean()

study4 = optuna.create_study(direction='maximize')
study4.optimize(objective, n_trials=10)

# results
study_result = pd.DataFrame({
    "cv_mean": [study4.best_value],
    "cv_std": [np.nan],
    "balanced_accuracy": [np.nan]

}, index=["XGB Bonus Tuning"])

results = pd.concat([results, study_result])

[I 2026-04-22 22:55:38,506] A new study created in memory with name: no-name-24fc5162-f2f4-4d44-9be6-0b65957e1e5c
[I 2026-04-22 22:56:43,234] Trial 0 finished with value: 0.9611904310242384 and parameters: {'max_delta_step': 3.709656727336607}. Best is trial 0 with value: 0.9611904310242384.
[I 2026-04-22 22:57:48,812] Trial 1 finished with value: 0.9611846868575895 and parameters: {'max_delta_step': 2.138673127049391}. Best is trial 0 with value: 0.9611904310242384.
[I 2026-04-22 22:58:56,140] Trial 2 finished with value: 0.961141287222661 and parameters: {'max_delta_step': 3.8146571162426834}. Best is trial 0 with value: 0.9611904310242384.
[I 2026-04-22 22:59:59,868] Trial 3 finished with value: 0.9613747765548251 and parameters: {'max_delta_step': 8.427180889297277}. Best is trial 3 with value: 0.9613747765548251.
[I 2026-04-22 23:01:05,106] Trial 4 finished with value: 0.961212121590344 and parameters: {'max_delta_step': 1.862010623041277}. Best is trial 3 with value: 0.9613747765

# Tuning LightGBM

**Score to Beat**: 0.9606

## Tuning 1

`min_child_weight, max_depth, min_split_loss`


In [57]:
def objective(trial):
    min_child_weight  = trial.suggest_float("min_child_weight", 1, 10)
    max_depth = trial.suggest_int('max_depth', 10, 25)
    min_split_loss = trial.suggest_float("min_split_loss", 0, 5)


    cls_optuna = LGBMClassifier(
                    random_state=42,
                    enable_categorical=True,
                    min_child_weight=min_child_weight,
                    max_depth=max_depth,
                    min_split_loss=min_split_loss,
                )

    pipe = Pipeline(
        [
            ("features", feature_pipe),
            ("model", cls_optuna)
        ]
    )

    cv_scores = cross_val_score(pipe, X_train, y_train, cv=skf, scoring='balanced_accuracy', n_jobs=-1)
    return cv_scores.mean()

study1a = optuna.create_study(direction='maximize')
study1a.optimize(objective, n_trials=10)

# results
study_result = pd.DataFrame({
    "cv_mean": [study1a.best_value],
    "cv_std": [np.nan],
    "balanced_accuracy": [np.nan]

}, index=["LGBM Tuning 1"])

results = pd.concat([results, study_result])

[I 2026-04-22 23:06:25,207] A new study created in memory with name: no-name-63772940-80dc-4394-9a98-f4cd8feaa96e
[I 2026-04-22 23:06:59,002] Trial 0 finished with value: 0.9613115135449206 and parameters: {'min_child_weight': 7.209034126632153, 'max_depth': 22, 'min_split_loss': 3.945006063619245}. Best is trial 0 with value: 0.9613115135449206.
[I 2026-04-22 23:07:26,746] Trial 1 finished with value: 0.9611597315394584 and parameters: {'min_child_weight': 5.824558777346403, 'max_depth': 23, 'min_split_loss': 2.205401152115173}. Best is trial 0 with value: 0.9613115135449206.
[I 2026-04-22 23:07:50,916] Trial 2 finished with value: 0.961205993654114 and parameters: {'min_child_weight': 4.082064364381548, 'max_depth': 23, 'min_split_loss': 0.4097303808392255}. Best is trial 0 with value: 0.9613115135449206.
[I 2026-04-22 23:08:11,136] Trial 3 finished with value: 0.9612216616822339 and parameters: {'min_child_weight': 6.624523030667861, 'max_depth': 10, 'min_split_loss': 1.343714029984

## Tuning 2

`bagging_fraction, colsample_bytree`


In [58]:
def objective(trial):
    bagging_fraction = trial.suggest_float("bagging_fraction", 0.6, 1.0)
    colsample_bytree = trial.suggest_float("colsample_bytree ", 0.3, 0.7)

    cls_optuna = LGBMClassifier(
                    random_state=42,
                    enable_categorical=True,
                    bagging_fraction=bagging_fraction,
                    colsample_bytree=colsample_bytree,
                )

    pipe = Pipeline(
        [
            ("features", feature_pipe),
            ("model", cls_optuna)
        ]
    )

    cv_scores = cross_val_score(pipe, X_train, y_train, cv=skf, scoring='balanced_accuracy', n_jobs=-1)
    return cv_scores.mean()

study2a = optuna.create_study(direction='maximize')
study2a.optimize(objective, n_trials=10)

# results
study_result = pd.DataFrame({
    "cv_mean": [study2a.best_value],
    "cv_std": [np.nan],
    "balanced_accuracy": [np.nan]

}, index=["LGBM Tuning 2"])

results = pd.concat([results, study_result])

[I 2026-04-22 23:09:59,632] A new study created in memory with name: no-name-b303108f-3f39-48f3-a290-f0ae50ae5139
[I 2026-04-22 23:10:16,665] Trial 0 finished with value: 0.9606403246704733 and parameters: {'bagging_fraction': 0.6466094233260622, 'colsample_bytree ': 0.6605522167706023}. Best is trial 0 with value: 0.9606403246704733.
[I 2026-04-22 23:10:32,787] Trial 1 finished with value: 0.9603903731358597 and parameters: {'bagging_fraction': 0.9553139395176866, 'colsample_bytree ': 0.5881463812247893}. Best is trial 0 with value: 0.9606403246704733.
[I 2026-04-22 23:10:49,183] Trial 2 finished with value: 0.959755374049494 and parameters: {'bagging_fraction': 0.7118323946711731, 'colsample_bytree ': 0.5412462282575244}. Best is trial 0 with value: 0.9606403246704733.
[I 2026-04-22 23:11:06,420] Trial 3 finished with value: 0.959755374049494 and parameters: {'bagging_fraction': 0.6330741731450739, 'colsample_bytree ': 0.5345128841049002}. Best is trial 0 with value: 0.96064032467047

## Tuning 3

`learning_rate, n_estimators`

In [59]:
def objective(trial):
    learning_rate  = trial.suggest_float("learning_rate ", 0.01, 0.4)
    n_estimators = trial.suggest_int("n_estimators ", 100, 750)


    cls_optuna = LGBMClassifier(
                    random_state=42,
                    enable_categorical=True,
                    learning_rate=learning_rate,
                    n_estimators=n_estimators,
                )

    pipe = Pipeline(
        [
            ("features", feature_pipe),
            ("model", cls_optuna)
        ]
    )

    cv_scores = cross_val_score(pipe, X_train, y_train, cv=skf, scoring='balanced_accuracy', n_jobs=-1)
    return cv_scores.mean()

study3a = optuna.create_study(direction='maximize')
study3a.optimize(objective, n_trials=10)

# results
study_result = pd.DataFrame({
    "cv_mean": [study3a.best_value],
    "cv_std": [np.nan],
    "balanced_accuracy": [np.nan]

}, index=["LGBM Tuning 3"])

results = pd.concat([results, study_result])

[I 2026-04-22 23:12:37,058] A new study created in memory with name: no-name-a9bb4bb0-4239-4cdb-81a8-50fe2ccc5df9
[I 2026-04-22 23:13:42,312] Trial 0 finished with value: 0.9612483963786751 and parameters: {'learning_rate ': 0.0371737906525387, 'n_estimators ': 496}. Best is trial 0 with value: 0.9612483963786751.
[I 2026-04-22 23:14:07,557] Trial 1 finished with value: 0.9613136762652712 and parameters: {'learning_rate ': 0.07233485063946479, 'n_estimators ': 198}. Best is trial 1 with value: 0.9613136762652712.
[I 2026-04-22 23:15:09,894] Trial 2 finished with value: 0.8289224032386308 and parameters: {'learning_rate ': 0.2948340241327789, 'n_estimators ': 543}. Best is trial 1 with value: 0.9613136762652712.
[I 2026-04-22 23:16:05,426] Trial 3 finished with value: 0.9371515440998884 and parameters: {'learning_rate ': 0.2472658078705277, 'n_estimators ': 431}. Best is trial 1 with value: 0.9613136762652712.
[I 2026-04-22 23:17:21,113] Trial 4 finished with value: 0.8106437246610747 a

## Bonus Tuning

In [60]:
def objective(trial):
    max_delta_step = trial.suggest_int("max_delta_step", 0, 10)

    cls_optuna = LGBMClassifier(
                    random_state=42,
                    enable_categorical=True,
                    max_delta_step=max_delta_step,
                    **study1a.best_params,
                    **study2a.best_params,
                    **study3a.best_params,
                )

    pipe = Pipeline(
        [
            ("features", feature_pipe),
            ("model", cls_optuna)
        ]
    )

    cv_scores = cross_val_score(pipe, X_train, y_train, cv=skf, scoring='balanced_accuracy', n_jobs=-1)
    return cv_scores.mean()

study4a = optuna.create_study(direction='maximize')
study4a.optimize(objective, n_trials=10)

# results
study_result = pd.DataFrame({
    "cv_mean": [study4a.best_value],
    "cv_std": [np.nan],
    "balanced_accuracy": [np.nan]

}, index=["LGBM Bonus Tuning"])

results = pd.concat([results, study_result])

[I 2026-04-22 23:22:17,312] A new study created in memory with name: no-name-d3583e23-35ea-4422-b144-795b12a7305b
[I 2026-04-22 23:22:36,614] Trial 0 finished with value: 0.9607687314111079 and parameters: {'max_delta_step': 1}. Best is trial 0 with value: 0.9607687314111079.
[I 2026-04-22 23:22:55,567] Trial 1 finished with value: 0.9613198125569777 and parameters: {'max_delta_step': 6}. Best is trial 1 with value: 0.9613198125569777.
[I 2026-04-22 23:23:13,818] Trial 2 finished with value: 0.9611460246056668 and parameters: {'max_delta_step': 3}. Best is trial 1 with value: 0.9613198125569777.
[I 2026-04-22 23:23:32,892] Trial 3 finished with value: 0.9610714674722161 and parameters: {'max_delta_step': 10}. Best is trial 1 with value: 0.9613198125569777.
[I 2026-04-22 23:23:52,417] Trial 4 finished with value: 0.9611064135834386 and parameters: {'max_delta_step': 8}. Best is trial 1 with value: 0.9613198125569777.
[I 2026-04-22 23:24:11,500] Trial 5 finished with value: 0.96101138008

# Results & Submission

In [61]:
results.sort_values("cv_mean", ascending=False)[["cv_mean"]]

,cv_mean
XGB Tuning 3,0.961715
Base XGB,0.961568
XGB Bonus Tuning,0.961375
LGBM Tuning 1,0.961350
LGBM Bonus Tuning,0.961320
LGBM Tuning 3,0.961314
XGB Tuning 1,0.961234
XGB Tuning 2,0.961141
LGBM Tuning 2,0.960640
Base LGBM,0.960589


In [76]:
# load training data
test_df = pd.read_csv(DATA_DIR / "test.csv", index_col="id")

cat_cols = test_df.select_dtypes(include=["str", "object"]).columns
test_df[cat_cols] = test_df[cat_cols].astype("category")

test_df.head()

,Soil_Type,Soil_pH,Soil_Moisture,Organic_Carbon,Electrical_Conductivity,Temperature_C,Humidity,Rainfall_mm,Sunlight_Hours,Wind_Speed_kmh,Crop_Type,Crop_Growth_Stage,Season,Irrigation_Type,Water_Source,Field_Area_hectare,Mulching_Used,Previous_Irrigation_mm,Region
id,,,,,,,,,,,,,,,,,,,
630000,Silt,6.36,26.19,0.59,2.81,17.83,30.24,1533.38,5.40,3.00,Maize,Sowing,Rabi,Canal,River,13.59,Yes,47.48,West
630001,Clay,5.87,9.88,1.18,3.26,21.18,78.07,576.05,7.22,15.88,Cotton,Sowing,Rabi,Drip,Reservoir,6.12,Yes,56.43,South
630002,Sandy,6.22,26.55,0.96,0.85,26.87,60.35,545.30,9.43,2.63,Wheat,Sowing,Kharif,Sprinkler,Reservoir,3.11,Yes,20.00,East
630003,Clay,7.68,53.58,0.83,0.55,41.74,36.05,1211.03,6.69,1.86,Maize,Harvest,Rabi,Canal,Groundwater,2.27,No,102.99,North
630004,Loamy,5.23,59.02,0.54,2.11,41.08,52.47,1321.91,4.11,5.71,Cotton,Sowing,Kharif,Canal,Groundwater,12.39,Yes,13.33,Central


In [79]:
# best model is
best_model = XGBClassifier(
                    random_state=42,
                    enable_categorical=True,
                    device="cuda:0",
                    tree_method="hist",
                    **study3.best_params
                )

final_model = Pipeline(
    [
        ("features", feature_pipe),
        ("xgb", best_model)
    ]
)

# fit on full dataset
final_model.fit(X, y_encoded)

# get and check preds
y_preds = pd.DataFrame(le.inverse_transform(final_model.predict(test_df)), index=test_df.index)
y_preds.head()

,0
id,
630000,Low
630001,Low
630002,Low
630003,Low
630004,Low


In [80]:
y_preds.to_csv(DATA_DIR / "tuned_boosting_preds.csv")

# Discussion

My default tuning method has been to throw everything into one large bucket and let gridsearch or optuna find the best combination.

I initially tried a "forward-feed" approach to tuning, where we move from tuning coarser hyperparameters to more granular hyperparameters, using the results from the previous tuning round. While there is still a chance that features between stages have interactions, I wanted to see if this could improve either model. Ultimately, this did not work well and saw an overall drop in performance.


---

**XGB** (0.9616 - Base)

| Tuning Round | Better Than Base? (n=10) |
|--------------|--------------------------|
| 1            | No                       |
| 2            | No                       |
| 3            | Yes                      |
| Bonus        | No                       |

- *Stage 1* was a very slight decrease.
- *Stage 2* was a slight decrease.
- *Stage 3* was a decent improvement!

---

**LGBM** (0.9606)

| Tuning Round | Better Than Base? (n=10) |
|--------------|--------------------------|
| 1            | Yes                      |
| 2            | Yes                      |
| 3            | Yes                      |
| Bonus        | Same                     |

- *Stage 1* was a decent increase.
- *Stage 2* same as stage 1
- *Stage 3* same as stage 1

# Takeaways

- avoid tuning everything all at once. especially if there aren't enough trials
- be patient

# Questions

- Do I need to use cross validation in optuna trials?
- What can knowledge of my dataset do to inform my tuning strategy? (eg imbalanced data, small total data, base model struggling to learn specific features)
- Are there any other scientific strategies to tuning (and tuning order?) or is it 99% an art?
- Is the forward-feed method used at all? Why or why not?
